# 문제 4 — 최종 시계열 분류 모델

6개 센서 × 15 step을 채널별 7개 통계(총 42개)로 변환하고 ExtraTrees로 분류합니다.

- 라벨: 0 정상, 1 공구 충돌, 2 파지 물체 충돌, 3 하부 진행 방해, 4 하부 충돌
- 평가: 클래스 `[0,1,2,3,4]` 고정 Macro-F1, `zero_division=0`
- 검증: 샘플 단위 StratifiedKFold 5-fold, shuffle=True, seed=42
- 선택 모델: 400 trees, min_samples_leaf=2, class_weight='balanced', max_features='sqrt'
- 기존 OOF Macro-F1: 0.80023694. 모델 선택에 사용한 검증 결과이며 독립 테스트 성능이 아닙니다.

CSV 4개가 있는 디렉터리에서 위에서 아래로 실행하세요. 다른 프로젝트 Python 파일에 의존하지 않습니다.
마지막 셀은 해당 디렉터리의 `model.joblib`과 `submission.csv`를 생성하거나 덮어씁니다.
필요 패키지: numpy, pandas, scikit-learn, joblib. 설치 환경의 패키지 버전을 아래에서 기록합니다.

In [1]:
from pathlib import Path
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

DATA_DIR = Path.cwd()
CLASSES = [0, 1, 2, 3, 4]
CHANNELS = ['force_x', 'force_y', 'force_z', 'torque_x', 'torque_y', 'torque_z']
SEED = 42
print({'python': platform.python_version(), 'numpy': np.__version__,
       'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'joblib': joblib.__version__})

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


{'python': '3.10.12', 'numpy': '1.26.4', 'pandas': '2.3.2', 'sklearn': '1.7.2', 'joblib': '1.6.0'}


## 1. 데이터와 샘플 대응 확인

In [2]:
train_series = pd.read_csv(DATA_DIR / 'train_series.csv')
labels = pd.read_csv(DATA_DIR / 'train_labels.csv')
test_series = pd.read_csv(DATA_DIR / 'test_series.csv')
template = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert labels.sample_id.is_unique and template.sample_id.is_unique
assert labels.label.isin(CLASSES).all()
assert set(labels.sample_id).isdisjoint(template.sample_id)
assert template.columns.tolist() == ['sample_id', 'label']
assert len(labels) == 120 and len(template) == 41
print('학습 샘플:', len(labels), '테스트 샘플:', len(template))
display(labels.label.value_counts().sort_index().rename('count').to_frame())

학습 샘플: 120 테스트 샘플: 41


,count
label,
0,33
1,19
2,33
3,16
4,19


## 2. 시퀀스 구성과 42개 특징

`ordered_ids` 순서와 라벨 순서를 일치시키고, 샘플 내부는 step 0~14로 정렬합니다.
채널마다 평균, 표본 표준편차(ddof=1), 최솟값, 최댓값, 마지막−처음,
연속 변화량 절댓값의 평균과 최댓값을 계산합니다. ID는 입력 특징에서 제외합니다.
ExtraTrees에는 스케일링을 적용하지 않습니다.

In [3]:
def pack_sequences(frame, ordered_ids):
    assert not frame.duplicated(['sample_id', 'step']).any()
    assert set(frame.sample_id) == set(ordered_ids)
    groups = frame.groupby('sample_id')
    sequences = []
    for sid in ordered_ids:
        group = groups.get_group(sid).sort_values('step')
        assert group.step.tolist() == list(range(15))
        sequences.append(group[CHANNELS].to_numpy(dtype=float))
    packed = np.stack(sequences)
    assert packed.shape == (len(ordered_ids), 15, 6)
    assert np.isfinite(packed).all()
    return packed


def extract_features(x):
    delta = np.abs(np.diff(x, axis=1))
    result = np.concatenate([
        x.mean(1), x.std(1, ddof=1), x.min(1), x.max(1),
        x[:, -1] - x[:, 0], delta.mean(1), delta.max(1)
    ], axis=1)
    assert result.shape == (len(x), 42) and np.isfinite(result).all()
    return result


X = extract_features(pack_sequences(train_series, labels.sample_id))
X_test = extract_features(pack_sequences(test_series, template.sample_id))
y = labels.label.to_numpy()
print('학습 특징:', X.shape, '테스트 특징:', X_test.shape)

학습 특징: (120, 42) 테스트 특징: (41, 42)


## 3. 동일한 5-fold OOF 검증
각 샘플은 자신을 학습하지 않은 모델로 한 번 예측합니다. 전체 OOF Macro-F1과 fold 평균은 별도로 표시합니다.

In [4]:
def macro_f1(true, predicted):
    return f1_score(true, predicted, labels=CLASSES, average='macro', zero_division=0)


model = ExtraTreesClassifier(
    n_estimators=400, min_samples_leaf=2, class_weight='balanced',
    max_features='sqrt', random_state=SEED, n_jobs=2
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.empty(len(y), dtype=int)
fold_scores = []
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X, y), 1):
    fold_model = clone(model).fit(X[fit_idx], y[fit_idx])
    oof[valid_idx] = fold_model.predict(X[valid_idx])
    fold_scores.append(macro_f1(y[valid_idx], oof[valid_idx]))
    print(f'Fold {fold}: {fold_scores[-1]:.6f}')

oof_score = macro_f1(y, oof)
print(f'OOF Macro-F1: {oof_score:.8f}')
print(f'Fold mean: {np.mean(fold_scores):.6f}, std: {np.std(fold_scores):.6f}')
display(pd.DataFrame({'label': CLASSES, 'F1': f1_score(
    y, oof, labels=CLASSES, average=None, zero_division=0)}))
display(pd.DataFrame(confusion_matrix(y, oof, labels=CLASSES),
                     index=pd.Index(CLASSES, name='true'),
                     columns=pd.Index(CLASSES, name='predicted')))

Fold 1: 0.622129


Fold 2: 0.854762


Fold 3: 0.838462


Fold 4: 0.831319


Fold 5: 0.750000
OOF Macro-F1: 0.80023694
Fold mean: 0.779334, std: 0.086558


,label,F1
0,0,0.925373
1,1,0.545455
2,2,0.732394
3,3,0.903226
4,4,0.894737


predicted,0,1,2,3,4
true,,,,,
0,31,1,1,0,0
1,0,9,10,0,0
2,3,4,26,0,0
3,0,0,0,14,2
4,0,0,1,1,17


## 4. 전체 학습·모델 저장·제출 파일 검증
테스트 라벨은 학습이나 모델 선택에 사용하지 않습니다. 제출 ID 순서는 제공된 양식을 그대로 유지합니다.

In [5]:
model.fit(X, y)
submission = template.copy()
submission['label'] = model.predict(X_test).astype(int)
assert submission.columns.tolist() == ['sample_id', 'label']
assert len(submission) == 41 and submission.sample_id.is_unique
assert submission.sample_id.equals(template.sample_id)
assert not submission.isna().any().any()
assert pd.api.types.is_integer_dtype(submission.label)
assert submission.label.isin(CLASSES).all()

artifact = {'model': model, 'representation': 'stats', 'channels': CHANNELS, 'classes': CLASSES}
joblib.dump(artifact, DATA_DIR / 'model.joblib')
loaded = joblib.load(DATA_DIR / 'model.joblib')
np.testing.assert_array_equal(loaded['model'].predict(X_test), submission.label)
submission.to_csv(DATA_DIR / 'submission.csv', index=False)
pd.testing.assert_frame_equal(pd.read_csv(DATA_DIR / 'submission.csv'), submission)
print('모델 재로딩 및 제출 형식 검증 완료:', DATA_DIR / 'submission.csv')
display(submission.head())
display(submission.label.value_counts().sort_index().rename('count').to_frame())

모델 재로딩 및 제출 형식 검증 완료: /tmp/problem04_notebook_sy8qd45f/submission.csv


,sample_id,label
0,SEG-1016,2
1,SEG-1157,2
2,SEG-1042,1
3,SEG-1081,2
4,SEG-1036,0


,count
label,
0,11
1,6
2,11
3,6
4,7
